# Day 9 — ILT 2: Watermark-Based Incremental Loading

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 1 — Need for Incremental Loading |
| **Duration** | 90 minutes |
| **Format** | Instructor-led — every query below is real and safe to run against `gbmart` |

### Learning Objectives
- Explain exactly how a cursor/watermark-based connector decides what's "new"
- Trace GlobalMart's real `orders_data_ingestion_cdc` pipeline configuration end-to-end
- Verify an incremental pipeline run using real Bronze-layer checks

---

## The Mechanism, in One Query

Every watermark-based incremental load boils down to one query shape, run on a schedule:

```sql
SELECT * FROM source_table WHERE updated_at > :last_watermark_value;
-- ... then, only after that batch is successfully written downstream:
UPDATE control_table SET last_watermark_value = MAX(updated_at) FROM this_batch;
```

Two things have to be true for this to work correctly:
1. **The source column must be reliably bumped on every change.** GlobalMart's Postgres `orders` table has a trigger, `trg_orders_updated_at`, that sets `updated_at = now()` automatically on every `UPDATE` — nobody has to remember to do it by hand.
2. **The watermark must only advance after a successful write.** If you advance the watermark before confirming the downstream write succeeded, a failed run silently loses that batch forever.

## GlobalMart's Real Pipeline Configuration

This is the actual, live Lakeflow Connect ingestion pipeline spec for `orders_data_ingestion_cdc` (built in Day 2) — not a hypothetical example:

```json
{
  "name": "orders_data_ingestion_cdc",
  "spec": {
    "catalog": "gbmart", "schema": "bronze", "serverless": true,
    "ingestion_definition": {
      "connection_name": "ecom_gbmart_conn",
      "source_type": "POSTGRESQL",
      "objects": [
        { "table": {
            "source_catalog": "postgres", "source_schema": "globalmart", "source_table": "orders",
            "destination_catalog": "gbmart", "destination_schema": "bronze", "destination_table": "orders",
            "table_configuration": {
              "primary_keys": ["orderid"],
              "query_based_connector_config": { "cursor_columns": ["updated_at"] }
        }}},
        { "table": {
            "source_catalog": "postgres", "source_schema": "globalmart", "source_table": "order_items",
            "destination_catalog": "gbmart", "destination_schema": "bronze", "destination_table": "order_items",
            "table_configuration": {
              "primary_keys": ["orderitemid"],
              "query_based_connector_config": { "cursor_columns": ["updated_at"] }
        }}}
      ]
    }
  }
}
```

Point out to the class: `primary_keys: ["orderid"]`, not `order_id`. Bronze columns coming from this Postgres source keep the source's own naming (`orderid`, `customerid`, `orderchannel`, `shippingdate`, `actualdeliverydate`) — no underscores. Day 5's Silver layer is what renames these to `order_id`, `customer_id`, etc. Bronze is raw-as-received, always.

In [ ]:
# Live, read-only. Confirms the real column naming discussed above — run this and
# actually look at the column names before moving on, don't just take the slide's word for it.
spark.sql("DESCRIBE gbmart.bronze.orders").select("col_name", "data_type").show(20, truncate=False)

## Verifying an Incremental Run Actually Happened

How do you *prove* the cursor picked up new changes, rather than just trusting the pipeline UI says "Succeeded"? Three checks, in order — this is the real verification pattern used against this pipeline.

In [ ]:
# Check 1 — total row counts right now. Note these down; if you make a change in
# Postgres and re-trigger the pipeline, these numbers are what you compare against.
spark.sql("""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM gbmart.bronze.orders
    UNION ALL
    SELECT 'order_items', COUNT(*) FROM gbmart.bronze.order_items
""").display()

In [ ]:
# Check 2 — Delta history shows a new version each time the pipeline runs and finds
# changes. If you re-trigger the pipeline and DESCRIBE HISTORY shows no new version,
# the cursor found nothing newer than its last watermark — which is expected if
# nothing actually changed in Postgres since the last run.
spark.sql("DESCRIBE HISTORY gbmart.bronze.orders") \
    .select("version", "timestamp", "operation") \
    .orderBy("version", ascending=False) \
    .show(5, truncate=False)

In [ ]:
# Check 3 — spot-check the most recently updated rows. Since the cursor is `updated_at`,
# the rows with the newest `updated_at` values are exactly the rows the pipeline would
# have picked up on its most recent run.
spark.sql("""
    SELECT orderid, customerid, orderchannel, updated_at
    FROM gbmart.bronze.orders
    ORDER BY updated_at DESC
    LIMIT 10
""").display()

## Proving the Blind Spot (Carried Forward from Day 2)

If a row were hard-deleted from `postgres.globalmart.orders` right now, the query above (`WHERE updated_at > last_watermark`) would simply never see it — there's no row left to have a newer `updated_at`. The row would sit in `gbmart.bronze.orders` forever, undeleted, with no error, no warning, and no signal that anything is wrong. This is exactly what Day 2's HOL had you prove hands-on by deleting a test row and watching it not sync.

**This is the one gap CDF closes** — which is exactly why GlobalMart uses CDF (not cursor-based watermarking) for the Bronze → Silver hop, covered next in ILT 3.